# formatting

> Display formatting utilities for dates, times, and filenames

In [ ]:
#| default_exp core.services.formatting

In [ ]:
#| export
from typing import Optional
from datetime import datetime
from pathlib import Path

## Date and Time

In [ ]:
#| export
def format_date(
    created_at: str  # ISO date string, Unix timestamp, or similar
) -> str:  # Formatted date for display
    """Format a date string for human-readable display (e.g., 'Jan 20, 2026')."""
    if not created_at:
        return "Unknown"
    
    # Try to parse as Unix timestamp first (float or int)
    try:
        timestamp = float(created_at)
        # Sanity check: timestamp should be reasonable (after year 2000, before year 2100)
        if 946684800 < timestamp < 4102444800:
            dt = datetime.fromtimestamp(timestamp)
            return dt.strftime("%b %d, %Y")  # e.g., "Jan 20, 2026"
    except (ValueError, TypeError, OSError):
        pass
    
    # Try to parse common ISO date formats
    date_formats = [
        "%Y-%m-%d %H:%M:%S.%f",  # SQLite with microseconds
        "%Y-%m-%d %H:%M:%S",     # SQLite without microseconds
        "%Y-%m-%dT%H:%M:%S.%f",  # ISO 8601 with microseconds
        "%Y-%m-%dT%H:%M:%S",     # ISO 8601 without microseconds
        "%Y-%m-%d",              # Date only
    ]
    
    for fmt in date_formats:
        try:
            dt = datetime.strptime(str(created_at).strip(), fmt)
            return dt.strftime("%b %d, %Y")  # e.g., "Jan 20, 2026"
        except ValueError:
            continue
    
    # Fallback: truncate if unparseable
    return str(created_at)[:16] if len(str(created_at)) > 16 else str(created_at)

In [ ]:
#| export
def format_time(
    seconds: Optional[float]  # Time in seconds
) -> str:  # Formatted time string (MM:SS)
    """Format seconds as MM:SS for display."""
    if seconds is None:
        return "--:--"
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    return f"{minutes:02d}:{secs:02d}"

def format_time_precise(
    seconds: Optional[float]  # Time in seconds
) -> str:  # Formatted time string (m:ss.s)
    """Format seconds as m:ss.s for sub-second display."""
    if seconds is None:
        return "-:--.-"
    minutes = int(seconds // 60)
    secs = seconds % 60
    return f"{minutes}:{secs:04.1f}"

## Display Names

In [ ]:
#| export
def format_audio_filename(
    audio_path: str  # Full path to audio file
) -> str:  # Shortened filename for display
    """Extract and format the filename from a path."""
    if not audio_path or audio_path == "Unknown":
        return "Unknown Source"
    return Path(audio_path).name

## Tests

In [ ]:
assert format_time(0) == "00:00"
assert format_time(65) == "01:05"
assert format_time(3661) == "61:01"
assert format_time(None) == "--:--"
print("format_time tests passed")

assert format_time_precise(6.6) == "0:06.6"
assert format_time_precise(9.8) == "0:09.8"
assert format_time_precise(65.25) == "1:05.2"
assert format_time_precise(0.0) == "0:00.0"
assert format_time_precise(None) == "-:--.-"
print("format_time_precise tests passed")

In [ ]:
assert format_date("") == "Unknown"
assert format_date(None) == "Unknown"
assert format_date("2026-01-20") == "Jan 20, 2026"
assert format_date("2026-01-20 14:30:00") == "Jan 20, 2026"
print("format_date tests passed")

format_date tests passed


In [ ]:
assert format_audio_filename("/home/user/audio/test.wav") == "test.wav"
assert format_audio_filename("Unknown") == "Unknown Source"
assert format_audio_filename("") == "Unknown Source"
print("format_audio_filename tests passed")

format_audio_filename tests passed


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()